In [7]:
from transformers import pipeline

model = pipeline("sentiment-analysis",model="ProsusAI/finbert")

result = model("Stock market is booming")

print(result)

/home/sudoverse/project/ai-stock/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 32871.49it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'label': 'neutral', 'score': 0.5849168300628662}]


In [3]:
import requests
url = ("https://www.nseindia.com/api/corporate-announcements?index=equities")
response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
data= response.json()
for d in data[:10]:
    print(d['sm_name'])
    
    


Latent View Analytics Limited
Ashoka Buildcon Limited
Zen Technologies Limited
Stallion India Fluorochemicals Limited
Nazara Technologies Limited
Shadowfax Technologies Limited
Lemon Tree Hotels Limited
Vishal Mega Mart Limited
Nazara Technologies Limited
Physicswallah Limited


In [11]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from transformers import BertTokenizer, BertForSequenceClassification, pipeline
from nsepython import nse_quote_ltp # Reliable for getting price data
import requests
url = ("https://www.nseindia.com/api/corporate-announcements?index=equities")
response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
data= response.json()

# 1. Load FinBERT Model
print("Loading AI Model...")
model_name = "ProsusAI/finbert"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)
nlp = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)


# 2. Process your 'data' list
results = []
cat_name = "Equities"

for entry in data:
    
    company = entry['sm_name']
    symbol = entry['symbol']
    dt = entry['sort_date']
    description = entry['attchmntText']
    
    print(f"Processing {company}...")

    # A. Get XBRL Sentiment
    
    sentiment = nlp(description)[0]
    
    # B. Get Stock Price 
    # Note: 'nse_quote_ltp' gets Current Price. 
    # For historical price at exact 'dt', you would need a paid API or historical dump.
    try:
        price = nse_quote_ltp(symbol)
    except:
        price = "N/A"

    results.append({
        "Company Name": company,
        "Broadcast Time": dt,
        "Stock Price": price,
        "FinBERT Label": sentiment['label'],
        "FinBERT Score": round(sentiment['score'], 4)
    })

# 3. Display as Table
if results:
    df = pd.DataFrame(results)
    display(df) # Shows table in Jupyter/Colab
    
    # Properly initialize the ExcelWriter to save the file
    with pd.ExcelWriter('NSE_Master_Analysis.xlsx') as writer:
        df.to_excel(writer, sheet_name=cat_name, index=False)
        
    print(f"\n  Finished {cat_name} sheet.")
    print("Processing Complete! Check 'NSE_Master_Analysis.xlsx'")
else:
    print("No data was found to process.")

Loading AI Model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 59449.62it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processing Latent View Analytics Limited...
Processing Ashoka Buildcon Limited...
Processing Zen Technologies Limited...
Processing Stallion India Fluorochemicals Limited...
Processing Nazara Technologies Limited...
Processing Shadowfax Technologies Limited...
Processing Lemon Tree Hotels Limited...
Processing Vishal Mega Mart Limited...
Processing Nazara Technologies Limited...
Processing Physicswallah Limited...
Processing Sab Events & Governance Now Media Limited...
Processing Acme Solar Holdings Limited...
Processing Power Grid Corporation of India Limited...
Processing Kesoram Industries Limited...
Processing Quadrant Future Tek Limited...
Processing VIP Industries Limited...
Processing Marsons Limited...
Processing One 97 Communications Limited...
Processing Atlanta Electricals Limited...
Processing SHREE CEMENT LIMITED...


,Company Name,Broadcast Time,Stock Price,FinBERT Label,FinBERT Score
0,Latent View Analytics Limited,2026-04-02 00:34:45,N/A,positive,0.5279
1,Ashoka Buildcon Limited,2026-04-01 23:25:09,N/A,neutral,0.9488
2,Zen Technologies Limited,2026-04-01 23:20:51,N/A,neutral,0.9448
3,Stallion India Fluorochemicals Limited,2026-04-01 23:10:59,N/A,neutral,0.5226
4,Nazara Technologies Limited,2026-04-01 23:09:43,N/A,neutral,0.9440
5,Shadowfax Technologies Limited,2026-04-01 23:09:15,N/A,neutral,0.9334
6,Lemon Tree Hotels Limited,2026-04-01 23:06:00,N/A,positive,0.7564
7,Vishal Mega Mart Limited,2026-04-01 23:01:42,N/A,neutral,0.9424
8,Nazara Technologies Limited,2026-04-01 23:00:53,N/A,neutral,0.9473
9,Physicswallah Limited,2026-04-01 22:56:31,N/A,neutral,0.9369



  Finished Equities sheet.
Processing Complete! Check 'NSE_Master_Analysis.xlsx'
